## Building A Chatbot
In this section we will design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation.


In [2]:
import os 
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [3]:
groq_api_key=os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq 

model=ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)


In [5]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage

from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")])

AIMessage(content="Hello Shivesh, nice to meet you. It's great to connect with a Chief AI Engineer like yourself. What kind of AI projects are you currently working on, or what aspects of AI engineering are you most interested in? I'm here to chat and help with any questions or topics you'd like to discuss.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 50, 'total_tokens': 115, 'completion_time': 0.181450838, 'completion_tokens_details': None, 'prompt_time': 0.004374363, 'prompt_tokens_details': None, 'queue_time': 0.223065284, 'total_time': 0.185825201}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea6e8-dd06-76e1-9a7c-357875777137-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 65, 'total_tokens': 115})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer"),
        AIMessage(content="Hello Shivesh! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Shivesh, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 121, 'total_tokens': 137, 'completion_time': 0.029871557, 'completion_tokens_details': None, 'prompt_time': 0.011696287, 'prompt_tokens_details': None, 'queue_time': 0.222272802, 'total_time': 0.041567844}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea6e8-dfe7-7a33-bb45-cfc3edfb8d62-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 16, 'total_tokens': 137})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input.

In [7]:
!pip install langchain_community

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory # chat history is imported from this library 
from langchain_core.runnables.history import RunnableWithMessageHistory

# every message that we are putting up needs to be added in the message history 


store={}

# When different different users are chatting over llm model how we are going to ensure that one session
# is completely different from the other session 


# session id will be used to differentiate one session from another
def  get_session_history(session_id:str)->BaseChatMessageHistory :# will create session id (string) .....return type will be chatmessage history
        if session_id not in store: 
            store[session_id]=ChatMessageHistory() # if session id not in store... means new session... so make new object of chatmessagehistory...
        return store[session_id]
    
    
with_message_history=RunnableWithMessageHistory(model,get_session_history) #for particular model and session_history this will give entire chat history for {model,session} pair 

            

D:\Temp\ipykernel_22456\787193593.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [11]:
response.content

"Nice to meet you, Shivesh. It's great to connect with a Chief AI Engineer like yourself. That's a fascinating field, and I'm sure you're working on some cutting-edge projects. What kind of AI applications are you currently focusing on? Are you working in a specific industry, such as healthcare, finance, or robotics? I'm all ears and would love to hear more about your work."

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [13]:
response.content

"Your name is Shivesh, and you're a Chief AI Engineer."

In [ ]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)
#this time we  have made a new session_id so it should not be able to remember my name 
response.content



"I don't know your name. I'm a large language model, I don't have the ability to know your personal information, including your name, unless you tell me. If you'd like to share your name, I'd be happy to chat with you and use it in our conversation."

In [17]:
response=with_message_history.invoke(
    [HumanMessage(content="My name is john ? ")],
    config=config1 # for this session id '{config : session_id : chat 1 }' we are interacting 
)
response.content

"Nice to meet you, John. However, I should let you know that I'm a large language model, I don't have the ability to verify or store personal information, so I'm taking your word for it. If you say your name is John, I'll use that in our conversation, but please keep in mind that I don't have any prior knowledge or records of your identity. How can I assist you today, John?"

In [ ]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)

# now this time it will show our name as john 
response.content

# In this way we can switch between context ....based on different session id ....

'You told me earlier that your name is John.'